# Phase 2 — Workplace-level modeling table

**Goal:** One row per workplace. Features from history, target from the future.

| Design choice | Decision (from log) |
|---|---|
| Grain | 1 row = 1 `WORKPLACE ID` (D002) |
| Feature window | 2017-01-01 → 2022-12-31 snapshot (D005, D008) |
| Target | Serious order in calendar year 2023 (D004) |
| Cohort | History **and** revisited in 2023 (~12,234 rows) (D008) |
| Schema | Stack 2017–2023 xlsx, core columns only (D007) |

**Leakage rules — memorize these:**
1. **Never** use 2023+ rows when computing features.
2. `days_since_last_visit` = days from last **2017–2022** visit to 2022-12-31.
3. Do **not** include workplaces without a 2023 visit (zeros = "not inspected," not "safe").
4. Do **not** stack 2024 yet — different column names and order-type strings (D007/D009).
5. `target_serious_2023` is the label — never put it in the feature matrix.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path("..").resolve()
DATA = ROOT / "data"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
CORE_COLS = [
    "WORKPLACE ID",
    "ORDER TYPE",
    "CASE TYPE",
    "FIELD VISIT DATE",
    "PRIMARY NAICS",
    "ORDER STATUS",
]
CUTOFF = pd.Timestamp("2022-12-31") # used for the cutoff 
HIST_YEARS = range(2017, 2023)   # 2017..2022 inclusive
TARGET_YEAR = 2023
SERIOUS_2023 = {
    "Stop Use/Stop Work Order",
    "Time Unknown Order",
}
STOP_WORK_HIST = {"Stop Use/Stop Work Order"}
TIME_UNKNOWN_HIST = {"Time Unknown Order"}

def load_year(year, cols=None):
    path = DATA / f"{year}_ohs_field_visit.xlsx"
    df = pd.read_excel(path, usecols=cols)
    df["source_year"] = year
    return df

def has_order(s):
    """Row has an enforcement action (not a visit-only row)."""
    return s.notna()

## Step 1 — Build the feature history (2017–2022)

Stack the historical years into one table and apply the hard date cutoff. Everything after 2022-12-31 is excluded to prevent leakage.

In [25]:
hist = pd.concat(
    [load_year(y, CORE_COLS) for y in HIST_YEARS],
    ignore_index=True,
)

hist["FIELD VISIT DATE"] = pd.to_datetime(hist["FIELD VISIT DATE"])

# Hard cutoff — anything later is leakage
hist = hist[hist["FIELD VISIT DATE"] <= CUTOFF].copy()
hist = hist.dropna(subset=["WORKPLACE ID"])
hist["WORKPLACE ID"] = hist["WORKPLACE ID"].astype(int)

print(f"History rows: {len(hist):,}")
print(f"History workplaces: {hist['WORKPLACE ID'].nunique():,}")
print(f"Date range: {hist['FIELD VISIT DATE'].min().date()} → {hist['FIELD VISIT DATE'].max().date()}")

History rows: 835,150
History workplaces: 169,092
Date range: 2017-01-02 → 2022-12-31


2023 Data will only be used for (a) cohort membership, (b) target- only and never for features

## Step 2 — Load the target year (2023)

Load 2023 and confirm row/workplace counts. This year supplies the label and defines cohort membership only — never features.

In [26]:
df23 = load_year(TARGET_YEAR, CORE_COLS)
df23["FIELD VISIT DATE"] = pd.to_datetime(df23["FIELD VISIT DATE"])
df23 = df23[
    (df23["FIELD VISIT DATE"] >= pd.Timestamp("2023-01-01"))
    & (df23["FIELD VISIT DATE"] <= pd.Timestamp("2023-12-31"))
].copy()
df23 = df23.dropna(subset=["WORKPLACE ID"])
df23["WORKPLACE ID"] = df23["WORKPLACE ID"].astype(int)

print(f"2023 rows: {len(df23):,}")
print(f"2023 workplaces: {df23['WORKPLACE ID'].nunique():,}")

2023 rows: 119,082
2023 workplaces: 30,911


In [27]:
hist_wps = set(hist["WORKPLACE ID"].unique())
wps_2023 = set(df23["WORKPLACE ID"].unique())
cohort = hist_wps & wps_2023
print(f"Cohort size: {len(cohort):,}") # expect ~12,234

Cohort size: 12,234


We will only evaluate and train on workplaces that the ministry revisited in 2023. Without this, most labels are false zeros from "never inspected again" and not low-risk (~0.6% positive vs ~8%)

# Creating the features 
## Step 3 — Engineer features

Aggregate the 2017–2022 history to one row per workplace: visit counts, enforcement counts, compliance rate, and recency. These are the model inputs.

In [ ]:
hist["PRIMARY NAICS"].value_counts()

PRIMARY NAICS
236110.0    219509
236220.0     72504
238160.0     22490
722512.0     20221
722511.0     18689
             ...  
414450.0         1
486990.0         1
112930.0         1
523130.0         1
911310.0         1
Name: count, Length: 878, dtype: int64

In [30]:
visit_key = ["WORKPLACE ID", "FIELD VISIT DATE"]

visits = (
    hist.groupby(visit_key, as_index=False)
    .agg(
        case_type=("CASE TYPE", "first"),      # same visit → same case type
        primary_naics=("PRIMARY NAICS", "first"),
    )
)

# order level non null rows 
orders = hist[has_order(hist["ORDER TYPE"])].copy()

# per workplace aggregates 
feat_visits = visits.groupby("WORKPLACE ID").agg(
    n_visits_5y=("FIELD VISIT DATE", "count"),
    n_investigations_5y=("case_type", lambda s: (s == "Investigation").sum()),
    days_since_last_visit=("FIELD VISIT DATE", lambda s: (CUTOFF - s.max()).days),
    primary_naics=("primary_naics", lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan),
)

feat_visits.head(5)




,n_visits_5y,n_investigations_5y,days_since_last_visit,primary_naics
WORKPLACE ID,,,,
27,24,11,17,212395.0
28,8,6,17,212395.0
32,1,1,837,112399.0
39,2,0,579,111211.0
41,4,4,628,212220.0


In [35]:
feat_orders = orders.groupby("WORKPLACE ID").agg(
    n_orders_5y=("ORDER TYPE", "count"),
    n_stop_work_5y=("ORDER TYPE", lambda s: s.isin(STOP_WORK_HIST).sum()),
    n_time_unknown_5y=("ORDER TYPE", lambda s: s.isin(TIME_UNKNOWN_HIST).sum()),
)

# Compliance: among rows with an order AND a status
orders_w_status = orders[orders["ORDER STATUS"].notna()]
complied = orders_w_status.groupby("WORKPLACE ID")["ORDER STATUS"].agg(
    n_with_status="count",
    n_complied=lambda s: (s == "Complied With").sum(),
)
complied["pct_complied_5y"] = complied["n_complied"] / complied["n_with_status"]
complied.columns

Index(['n_with_status', 'n_complied', 'pct_complied_5y'], dtype='str')

In [33]:
features = feat_visits.join(feat_orders, how="left").join(
    complied[["pct_complied_5y"]], how="left"
)
features["investigation_ratio"] = (
    features["n_investigations_5y"] / features["n_visits_5y"]
)
# Fill workplaces with visits but zero orders
for col in ["n_orders_5y", "n_stop_work_5y", "n_time_unknown_5y"]:
    features[col] = features[col].fillna(0).astype(int)
features.head()

,n_visits_5y,n_investigations_5y,days_since_last_visit,primary_naics,n_orders_5y,n_stop_work_5y,n_time_unknown_5y,pct_complied_5y,investigation_ratio
WORKPLACE ID,,,,,,,,,
27,24,11,17,212395.0,19,0,0,1.0,0.458333
28,8,6,17,212395.0,8,0,0,1.0,0.750000
32,1,1,837,112399.0,5,0,0,1.0,1.000000
39,2,0,579,111211.0,6,0,0,1.0,0.000000
41,4,4,628,212220.0,1,0,0,1.0,1.000000


In [36]:
target = (df23.groupby("WORKPLACE ID")["ORDER TYPE"]
    .apply(lambda s: s.isin(SERIOUS_2023).any())
    .rename("target_serious_2023")
    .astype(int)
)

print(f"2023 serious rate (all visited): {target.mean():.1%}")
print(f"2023 serious positives: {target.sum():,}")

2023 serious rate (all visited): 9.3%
2023 serious positives: 2,866


In [41]:
modeling_table = (
    features.join(target, how = "inner") # only workplaces with 2023 label
    .loc[lambda d: d.index.isin(cohort)]
    .reset_index()
    .rename(columns={"WORKPLACE ID": "workplace_id"})
)

# Re-attach workplace_id as column if index name differs
if "workplace_id" not in modeling_table.columns:
    modeling_table = modeling_table.rename(columns={"index": "workplace_id"})

assert len(modeling_table) > 12_000, "Cohort too small — check year filters"
assert abs(len(modeling_table) - 12_234) < 500, f"Expected ~12,234 rows, got {len(modeling_table)}"
assert modeling_table["target_serious_2023"].mean() > 0.05, "Positive rate suspiciously low"
assert modeling_table["target_serious_2023"].mean() < 0.15, "Positive rate suspiciously high"
assert modeling_table["workplace_id"].is_unique
print(f"Rows: {len(modeling_table):,}")
print(f"Target rate: {modeling_table['target_serious_2023'].mean():.1%}")
print(f"Positives: {modeling_table['target_serious_2023'].sum():,}")
modeling_table.describe()

Rows: 12,234
Target rate: 8.0%
Positives: 974


,workplace_id,n_visits_5y,n_investigations_5y,days_since_last_visit,primary_naics,n_orders_5y,n_stop_work_5y,n_time_unknown_5y,pct_complied_5y,investigation_ratio,target_serious_2023
count,1.223400e+04,12234.000000,12234.000000,12234.000000,12230.000000,12234.000000,12234.000000,12234.000000,8562.000000,12234.000000,12234.000000
mean,1.387281e+06,5.546183,3.558280,522.681625,450866.891823,6.786251,0.421775,0.583538,0.962969,0.507004,0.079614
std,8.587687e+05,8.921826,7.643913,554.077674,193665.699587,18.902803,2.361501,3.746320,0.097423,0.433241,0.270706
min,2.700000e+01,1.000000,0.000000,0.000000,111120.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.716642e+05,1.000000,0.000000,99.000000,311224.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
50%,1.693734e+06,3.000000,1.000000,304.000000,445110.000000,3.000000,0.000000,0.000000,1.000000,0.537887,0.000000
75%,2.180819e+06,6.000000,4.000000,761.000000,621610.000000,8.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,2.342015e+06,241.000000,228.000000,2186.000000,914110.000000,771.000000,95.000000,171.000000,1.000000,1.000000,1.000000


In [42]:
out_csv = OUT / "modeling_table.csv"
modeling_table.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")
print(modeling_table["target_serious_2023"].value_counts(normalize=True))

Saved /Users/user/Desktop/Technical Project/MLITSD-DS-Project/data/processed/modeling_table.csv
target_serious_2023
0    0.920386
1    0.079614
Name: proportion, dtype: float64
